# Лабораторное задание. Алгоритм RSA

**Выполнил** Зелёный Артём Сергеевич  
**Группа** М8О-209СВ-25  
**Вариант** 10

**Источник.** «ПЗ - Шифрование Часть 2.pptx», слайды 11–16. В презентации это задание подписано «ЛР-4». Для расчёта используем числовой пример со слайдов и отдельный контрольный набор параметров.

## 1. Генерация ключей

Для различных простых $p,q$ вычисляем $n=pq$ и $\varphi(n)=(p-1)(q-1)$. Выбираем $1<e<\varphi(n)$, взаимно простое с $\varphi(n)$. Расширенный алгоритм Евклида находит коэффициенты Безу $x,y$ такие, что $xe+y\varphi(n)=1$. Тогда $d=x\bmod\varphi(n)$. Пары $(e,n)$ и $(d,n)$ задают открытый и закрытый ключи.

In [1]:
from math import isqrt

def is_prime(value):
    return value >= 2 and all(value % divisor for divisor in range(2, isqrt(value) + 1))

def extended_gcd(a, b):
    old_r, r = a, b
    old_x, x = 1, 0
    old_y, y = 0, 1
    while r:
        quotient = old_r // r
        old_r, r = r, old_r - quotient * r
        old_x, x = x, old_x - quotient * x
        old_y, y = y, old_y - quotient * y
    return old_r, old_x, old_y

def make_keys(p, q, e):
    if p == q or not is_prime(p) or not is_prime(q):
        raise ValueError('p и q должны быть различными простыми числами')
    modulus = p * q
    phi = (p - 1) * (q - 1)
    if not 1 < e < phi:
        raise ValueError('Ожидается 1 < e < φ(n)')
    divisor, coefficient, _ = extended_gcd(e, phi)
    if divisor != 1:
        raise ValueError('e и φ(n) должны быть взаимно просты')
    return (e, modulus), (coefficient % phi, modulus)

## 2. Шифрование и расшифрование

Для блока $0\le m<n$ шифрование вычисляет $c=m^e\bmod n$, а расшифрование вычисляет $m=c^d\bmod n$. Быстрое возведение в степень выполняет умножения по модулю и на каждом шаге делит показатель на 2.

Корректность проверяется отдельно по модулям $p$ и $q$. Если $m$ кратно одному из простых модулей, обе части сравнения равны нулю по нему. Для остальных остатков применяется малая теорема Ферма, поскольку $ed-1$ кратно $p-1$ и $q-1$. Китайская теорема об остатках даёт $m^{ed}\equiv m\pmod n$.

In [2]:
def mod_exp(base, exponent, modulus):
    result = 1
    base %= modulus
    while exponent:
        if exponent & 1:
            result = result * base % modulus
        base = base * base % modulus
        exponent >>= 1
    return result

def transform(block, key):
    exponent, modulus = key
    if not 0 <= block < modulus:
        raise ValueError('Блок должен лежать в диапазоне 0 ≤ блок < n')
    return mod_exp(block, exponent, modulus)

def encrypt(message, public_key):
    return transform(message, public_key)

def decrypt(ciphertext, private_key):
    return transform(ciphertext, private_key)

## 3. Числовые проверки

В примере из презентации $p=3$, $q=7$, $e=5$ и $m=19$. Расширенный алгоритм Евклида даёт $d=5$ как наименьший положительный представитель. Указанное в презентации значение $17$ принадлежит тому же классу вычетов по модулю 12.

In [3]:
public_key, private_key = make_keys(3, 7, 5)
encrypted = encrypt(19, public_key)
restored = decrypt(encrypted, private_key)
print('Открытый ключ —', public_key)
print('Закрытый ключ —', private_key)
print(f'Сообщение 19 → шифротекст {encrypted} → исходное число {restored}')
assert encrypted == 10 and restored == 19
assert decrypt(encrypted, (17, 21)) == restored

for p, q, e in [(3, 7, 5), (61, 53, 17)]:
    public_key, private_key = make_keys(p, q, e)
    modulus = public_key[1]
    assert (e * private_key[0]) % ((p - 1) * (q - 1)) == 1
    for message in range(modulus):
        encrypted = encrypt(message, public_key)
        assert encrypted == pow(message, e, modulus)
        assert decrypt(encrypted, private_key) == message
    print(f'n = {modulus}. Проверены все {modulus} остатков.')

Открытый ключ — (5, 21)
Закрытый ключ — (5, 21)
Сообщение 19 → шифротекст 10 → исходное число 19
n = 21. Проверены все 21 остатков.
n = 3233. Проверены все 3233 остатков.


## Вывод

Реализованы вычисление ключей, шифрование и расшифрование числовых блоков. Пример из презентации дал шифротекст 10 и восстановил число 19. Полный перебор блоков для двух небольших модулей проверил обратимость, в том числе для блоков, кратных $p$ или $q$. Используется учебная формула RSA без схемы дополнения.